# 🚀 AI-Edited Photo Detection — EfficientNetV2B2 (SAGI-D Dataset)

**Dataset:** SAGI-D (Semantically Aligned and Guided Inpainting Dataset) on Kaggle
- **Real Images**: Original camera photos (sourced from RAISE & COCO datasets)
- **Fake Images**: The same photos edited using various state-of-the-art AI Inpainting models (Stable Diffusion, etc.)

**Goal**: Train a model to classify whether a photo is a **real camera-original** OR an **AI-edited/inpainted** photo.

---
### ⚡ Quick Start
1. `Runtime` → `Change runtime type` → Select **T4 GPU**
2. Run **Cell 1** (Mount Drive + GPU check)
3. Run **Cell 2** (Upload `kaggle.json` and download SAGI-D dataset)
4. Run **Cell 3** (Organize & Balance the Dataset)
5. Run **All remaining cells** or `Runtime → Run all`

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1: Mount Google Drive + Check GPU + Enable Mixed Precision
# ─────────────────────────────────────────────────────────
import os, sys

# Mount Google Drive (for crash-safe checkpoints)
from google.colab import drive
drive.mount('/content/drive')

# Create project folder on Drive
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/RAISE_DiQuID_Model'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f'✅ Google Drive mounted. Project folder: {DRIVE_PROJECT_DIR}')

# Check GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'\n🖥️  GPU Available: {len(gpus) > 0}')
if gpus:
    gpu_info = tf.config.experimental.get_device_details(gpus[0])
    print(f'   GPU: {gpu_info.get("device_name", "Unknown")}')
    # Enable Mixed Precision for ~2x speed boost
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print('   Mixed Precision (float16): ENABLED ✅')
else:
    print('⚠️  WARNING: No GPU found! Training will be very slow.')
    print('   Go to Runtime → Change runtime type → Select T4 GPU')

print(f'\n📦 TensorFlow version: {tf.__version__}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2: Download SAGI-D (DiQuID) Dataset via Kaggle API
# ─────────────────────────────────────────────────────────
import os

DATASET_DIR = '/content/sagi_dataset'

if os.path.exists(os.path.join(DATASET_DIR, 'train')) or os.path.exists(os.path.join(DATASET_DIR, 'val')):
    print('✅ SAGI-D Dataset already downloaded. Skipping.')
else:
    print('📥 Downloading SAGI-D dataset via Kaggle...')
    print('   Please upload your kaggle.json file when prompted.')
    print('   (Get it from: kaggle.com → Profile → Settings → API → Create New Token)')
    print()

    # Upload kaggle.json
    from google.colab import files
    uploaded = files.upload()

    # Set up Kaggle credentials
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

    # Download dataset
    os.makedirs(DATASET_DIR, exist_ok=True)
    print('\n📥 Downloading SAGI-D dataset (this might take some time)...')
    !kaggle datasets download -d giakop/sagi-d -p {DATASET_DIR}

    print('\n📦 Extracting dataset...')
    !unzip -q {DATASET_DIR}/sagi-d.zip -d {DATASET_DIR}
    print('✅ Dataset successfully extracted!')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3: Organize Dataset into Balanced Train/Val/Test Splits
# ─────────────────────────────────────────────────────────
# SAGI-D structure is:
# sagi_dataset/{split}/{origin_dataset}/original/  <-- Real images
# sagi_dataset/{split}/{origin_dataset}/{inpainting_model}/  <-- Fake images
#
# We will extract these and save them cleanly into:
# /content/dataset/train/real/
# /content/dataset/train/fake/
# ─────────────────────────────────────────────────────────
import os, shutil, random
from pathlib import Path
from tqdm import tqdm

SRC_DIR = '/content/sagi_dataset'
DST_DIR = '/content/dataset'
SEED = 42
random.seed(SEED)

shutil.rmtree(DST_DIR, ignore_errors=True)
os.makedirs(DST_DIR, exist_ok=True)

for split in ['train', 'val', 'test']:
    split_src = os.path.join(SRC_DIR, split)
    if not os.path.exists(split_src):
        # Sometimes 'val' is named 'valid'
        if split == 'val' and os.path.exists(os.path.join(SRC_DIR, 'valid')):
            split_src = os.path.join(SRC_DIR, 'valid')
        else:
            continue

    print(f'\n📂 Processing {split} split...')
    
    real_dest = os.path.join(DST_DIR, split, 'real')
    fake_dest = os.path.join(DST_DIR, split, 'fake')
    os.makedirs(real_dest, exist_ok=True)
    os.makedirs(fake_dest, exist_ok=True)

    # Traverse the origin folders (coco, raise, openimages)
    for origin in os.listdir(split_src):
        origin_path = os.path.join(split_src, origin)
        if not os.path.isdir(origin_path):
            continue
        
        original_folder = os.path.join(origin_path, 'original')
        if not os.path.exists(original_folder):
            continue

        # Get all real images in this folder
        real_files = [f for f in os.listdir(original_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        # Find inpainting model folders in this origin
        inpainting_folders = [
            os.path.join(origin_path, d) 
            for d in os.listdir(origin_path) 
            if os.path.isdir(os.path.join(origin_path, d)) and d != 'original'
        ]
        
        if not inpainting_folders:
            continue

        print(f'   - Origin: {origin} | Found {len(real_files)} real images')

        # For each real image, copy it, and copy exactly ONE matching fake version 
        # from a randomly chosen inpainting model folder to keep classes perfectly balanced.
        for f in tqdm(real_files, desc=f'Copying {origin}'):
            # Copy Real
            src_real = os.path.join(original_folder, f)
            dst_real = os.path.join(real_dest, f'{origin}_{f}')
            shutil.copy2(src_real, dst_real)

            # Copy Fake
            # Pick a random model folder containing the same filename
            random.shuffle(inpainting_folders)
            copied_fake = False
            for model_folder in inpainting_folders:
                src_fake = os.path.join(model_folder, f)
                if os.path.exists(src_fake):
                    dst_fake = os.path.join(fake_dest, f'{origin}_{f}')
                    shutil.copy2(src_fake, dst_fake)
                    copied_fake = True
                    break
            
            # If for some reason the fake image is missing from models, remove the real one to keep balance
            if not copied_fake:
                if os.path.exists(dst_real):
                    os.remove(dst_real)
    # Print class counts
    real_count = len(os.listdir(real_dest))
    fake_count = len(os.listdir(fake_dest))
    print(f'   📊 Split {split} summary: {real_count} Real, {fake_count} Fake')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4: Configuration — Paths & Hyperparameters
# ─────────────────────────────────────────────────────────
import os

# ── Paths ─────────────────────────────────────────────────
BASE_DIR         = '/content/dataset'
DRIVE_DIR        = '/content/drive/MyDrive/RAISE_DiQuID_Model'

# All checkpoints save to Google Drive → survives Colab disconnects
CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, 'checkpoints')
MODEL_SAVE_PATH  = os.path.join(DRIVE_DIR, 'raise_diquid_efficientnetv2b2.keras')
STATE_FILE       = os.path.join(CHECKPOINT_DIR, 'training_state.json')
HISTORY_FILE     = os.path.join(DRIVE_DIR, 'training_history.json')
PLOT_PATH        = os.path.join(DRIVE_DIR, 'training_history.png')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────
# EfficientNetV2B2: Good balance of speed and accuracy
# Better than B0 for detecting subtle AI artifacts
IMAGE_SIZE    = (260, 260)   # EfficientNetV2B2 native size
BATCH_SIZE    = 32           # Smaller batch for larger image size
SEED          = 42

# 3-Phase Training
EPOCHS_HEAD   = 10           # Phase 1: Train head only (frozen base)
EPOCHS_FINE1  = 15           # Phase 2: Unfreeze top 50 layers
EPOCHS_FINE2  = 15           # Phase 3: Unfreeze top 100 layers

# Learning rates
LR_HEAD       = 1e-3         # Phase 1
LR_FINE1      = 3e-5         # Phase 2
LR_FINE2      = 1e-5         # Phase 3

print('✅ Configuration loaded:')
print(f'   Model         : EfficientNetV2B2')
print(f'   Image Size    : {IMAGE_SIZE}')
print(f'   Batch Size    : {BATCH_SIZE}')
print(f'   Phase 1 epochs: {EPOCHS_HEAD}')
print(f'   Phase 2 epochs: {EPOCHS_FINE1}')
print(f'   Phase 3 epochs: {EPOCHS_FINE2}')
print(f'   Total epochs  : {EPOCHS_HEAD + EPOCHS_FINE1 + EPOCHS_FINE2}')
print(f'\n   Drive save dir: {DRIVE_DIR}')
print(f'   Final model   : {MODEL_SAVE_PATH}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5: Helper Functions (Resume + State Tracking)
# ─────────────────────────────────────────────────────────
import json, glob, re
import tensorflow as tf

def find_latest_checkpoint(phase):
    """Find latest checkpoint for a given phase."""
    pattern = os.path.join(CHECKPOINT_DIR, f'raise_diquid_{phase}_epoch_*.keras')
    files   = glob.glob(pattern)
    if not files:
        return None, 0
    def get_epoch(f):
        m = re.search(r'epoch_(\d+)\.keras', os.path.basename(f))
        return int(m.group(1)) if m else 0
    files.sort(key=get_epoch)
    latest = files[-1]
    return latest, get_epoch(latest)

def load_state():
    """Load training state from Google Drive."""
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            state = json.load(f)
        print(f'📂 Resumed state: {state}')
        return state
    print('🆕 No previous state. Starting from scratch.')
    return {'phase': 'p1', 'last_epoch': 0}

def save_state(phase, epoch, val_acc=None, val_loss=None):
    """Save training state to Google Drive."""
    state = {'phase': phase, 'last_epoch': epoch}
    if val_acc  is not None: state['best_val_acc']  = round(float(val_acc),  6)
    if val_loss is not None: state['best_val_loss'] = round(float(val_loss), 6)
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)

class DriveStateLogger(tf.keras.callbacks.Callback):
    """Saves state to Google Drive after every epoch — disconnect-safe."""
    def __init__(self, phase, initial_epoch=0):
        super().__init__()
        self.phase         = phase
        self.initial_epoch = initial_epoch

    def on_epoch_end(self, epoch, logs=None):
        abs_epoch = self.initial_epoch + epoch + 1
        val_acc   = logs.get('val_accuracy') if logs else None
        val_loss  = logs.get('val_loss')     if logs else None
        save_state(self.phase, abs_epoch, val_acc, val_loss)
        acc_str   = f'{val_acc:.4f}' if val_acc else 'N/A'
        print(f'  💾 Drive saved → phase={self.phase}, epoch={abs_epoch}, val_acc={acc_str}')

print('✅ Helper functions loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6: Data Loading + Preprocessing Pipeline
# ─────────────────────────────────────────────────────────
import tensorflow as tf
import gc

AUTOTUNE = tf.data.AUTOTUNE

# ── Data Augmentation ──────────────────────────────────
# IMPORTANT: We keep augmentation MILD.
# AI-edited photos have subtle pixel-level artifacts.
# Heavy augmentation (color/brightness changes) would destroy those artifacts
# and make the model unable to tell real from fake.
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.03),         # Very mild rotation
    tf.keras.layers.RandomZoom(0.03),             # Very mild zoom
    tf.keras.layers.RandomTranslation(0.03, 0.03),
# NO RandomBrightness / RandomContrast — preserves AI artifact signals
], name='augmentation')

# ── Load datasets ──────────────────────────────────────
print('📂 Loading datasets...')

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'train'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'val'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_DIR, 'test'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

class_names = train_ds.class_names
print(f'✅ Classes: {class_names}')  # Should be: [fake, real]
print(f'   Class 0 = {class_names[0]} | Class 1 = {class_names[1]}')
print(f'   (Model outputs 0=fake, 1=real)')

# Count
train_count = sum(1 for _ in train_ds.unbatch())
val_count   = sum(1 for _ in val_ds.unbatch())
test_count  = sum(1 for _ in test_ds.unbatch())
print(f'\n   Train: {train_count:,} images')
print(f'   Val  : {val_count:,} images')
print(f'   Test : {test_count:,} images')

# ── Pipeline Optimization ─────────────────────────────
train_ds = (
    train_ds
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

gc.collect()
print('✅ Data pipeline ready.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7: Build Model — EfficientNetV2B2 + Custom Head
# ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers

def build_model():
    base_model = tf.keras.applications.EfficientNetV2B2(
        include_top=False,
        weights='imagenet',
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        pooling=None
    )
    base_model.trainable = False  # Frozen in Phase 1

    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

    # EfficientNetV2 has built-in preprocessing
    x = tf.keras.applications.efficientnet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)

    # ── Custom Classifier Head ────────────────────────
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)          # Higher dropout — smaller dataset
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    # Sigmoid output for binary: 0=fake, 1=real
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs, name='RAISE_DiQuID_Detector')
    return model, base_model

model, base_model = build_model()

trainable = sum(1 for l in model.layers if l.trainable)
total     = len(model.layers)
print(f'✅ Model built: EfficientNetV2B2')
print(f'   Total layers    : {total}')
print(f'   Trainable layers: {trainable} (head only, base frozen)')
print(f'   Total params    : {model.count_params():,}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8: PHASE 1 — Train Classifier Head Only
# ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import callbacks
import gc

current_state    = load_state()
p1_ckpt, _       = find_latest_checkpoint('p1')
p1_initial_epoch = (
    current_state['last_epoch'] if current_state['phase'] == 'p1'
    else EPOCHS_HEAD
)

history_p1 = None

if p1_initial_epoch >= EPOCHS_HEAD:
    print(f'✅ Phase 1 already complete ({p1_initial_epoch}/{EPOCHS_HEAD} epochs).')
    if p1_ckpt:
        model.load_weights(p1_ckpt)
        print(f'   Loaded: {os.path.basename(p1_ckpt)}')
else:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
        # Label smoothing: prevents overconfidence, helps generalize
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    if p1_ckpt and p1_initial_epoch > 0:
        model.load_weights(p1_ckpt)
        print(f'↩️  Resuming Phase 1 from epoch {p1_initial_epoch}')
    else:
        print('🆕 Starting Phase 1 from scratch.')

    cb_p1 = [
        callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'raise_diquid_p1_epoch_{epoch:02d}.keras'),
            save_best_only=True, monitor='val_accuracy', mode='max', verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=5,
            restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3,
            min_lr=1e-6, verbose=1
        ),
        DriveStateLogger(phase='p1', initial_epoch=p1_initial_epoch),
    ]

    print(f'\n=== PHASE 1: Train Head Only ===')
    print(f'    LR={LR_HEAD} | Epochs: {p1_initial_epoch+1} → {EPOCHS_HEAD}')
    history_p1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_HEAD,
        initial_epoch=p1_initial_epoch,
        callbacks=cb_p1
    )

    save_state('p2', 0)
    gc.collect()
    print('✅ Phase 1 complete.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9: PHASE 2 — Fine-tune Top 50 Layers
# ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import callbacks
import gc

# Unfreeze top 50 layers of EfficientNetV2B2
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'🔓 Unfrozen (Phase 2): {unfrozen}/{len(base_model.layers)} layers')

steps_per_epoch = len(train_ds)
total_steps_p2  = EPOCHS_FINE1 * steps_per_epoch
cosine_decay_p2 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_FINE1,
    decay_steps=total_steps_p2,
    alpha=1e-6
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_decay_p2),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

current_state    = load_state()
p2_ckpt, _       = find_latest_checkpoint('p2')
p2_initial_epoch = (
    current_state['last_epoch'] if current_state['phase'] == 'p2'
    else (EPOCHS_FINE1 if current_state['phase'] == 'p3' else 0)
)

history_p2 = None

if p2_initial_epoch >= EPOCHS_FINE1:
    print(f'✅ Phase 2 already complete ({p2_initial_epoch}/{EPOCHS_FINE1} epochs).')
    if p2_ckpt:
        model.load_weights(p2_ckpt)
        print(f'   Loaded: {os.path.basename(p2_ckpt)}')
else:
    if p2_ckpt and p2_initial_epoch > 0:
        model.load_weights(p2_ckpt)
        print(f'↩️  Resuming Phase 2 from epoch {p2_initial_epoch}')
    else:
        print('🆕 Starting Phase 2 from scratch.')

    cb_p2 = [
        callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'raise_diquid_p2_epoch_{epoch:02d}.keras'),
            save_best_only=True, monitor='val_accuracy', mode='max', verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=5,
            restore_best_weights=True, verbose=1
        ),
        DriveStateLogger(phase='p2', initial_epoch=p2_initial_epoch),
    ]

    print(f'\n=== PHASE 2: Fine-tune Top 50 Layers ===')
    print(f'    LR=CosineDecay({LR_FINE1}) | Epochs: {p2_initial_epoch+1} → {EPOCHS_FINE1}')
    history_p2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE1,
        initial_epoch=p2_initial_epoch,
        callbacks=cb_p2
    )

    save_state('p3', 0)
    gc.collect()
    print('✅ Phase 2 complete.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 10: PHASE 3 — Deep Fine-tune (Top 100 Layers)
# ─────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import callbacks
import gc

# Unfreeze top 100 layers
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'🔓 Unfrozen (Phase 3): {unfrozen}/{len(base_model.layers)} layers')

steps_per_epoch  = len(train_ds)
total_steps_p3   = EPOCHS_FINE2 * steps_per_epoch

cosine_decay_p3 = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LR_FINE2,
    first_decay_steps=total_steps_p3 // 2,
    t_mul=1.5,
    m_mul=0.9,
    alpha=1e-7
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=cosine_decay_p3,
        weight_decay=1e-4   # L2 regularization
    ),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

current_state    = load_state()
p3_ckpt, _       = find_latest_checkpoint('p3')
p3_initial_epoch = (
    current_state['last_epoch'] if current_state['phase'] == 'p3' else 0
)

history_p3 = None

if p3_initial_epoch >= EPOCHS_FINE2:
    print(f'✅ Phase 3 already complete ({p3_initial_epoch}/{EPOCHS_FINE2} epochs).')
    if p3_ckpt:
        model.load_weights(p3_ckpt)
        print(f'   Loaded: {os.path.basename(p3_ckpt)}')
else:
    if p3_ckpt and p3_initial_epoch > 0:
        model.load_weights(p3_ckpt)
        print(f'↩️  Resuming Phase 3 from epoch {p3_initial_epoch}')
    else:
        print('🆕 Starting Phase 3 from scratch.')

    cb_p3 = [
        callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, 'raise_diquid_p3_epoch_{epoch:02d}.keras'),
            save_best_only=True, monitor='val_accuracy', mode='max', verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy', patience=6,
            restore_best_weights=True, verbose=1
        ),
        DriveStateLogger(phase='p3', initial_epoch=p3_initial_epoch),
    ]

    print(f'\n=== PHASE 3: Deep Fine-tune (Top 100 Layers) ===')
    print(f'    LR=CosineDecayRestarts({LR_FINE2}) | Epochs: {p3_initial_epoch+1} → {EPOCHS_FINE2}')
    history_p3 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE2,
        initial_epoch=p3_initial_epoch,
        callbacks=cb_p3
    )

    save_state('done', EPOCHS_FINE2)
    gc.collect()
    print('✅ Phase 3 complete.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 11: Evaluation on Test Set + Confusion Matrix
# ─────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print('📊 Evaluating on test set...')
results = model.evaluate(test_ds, verbose=1)

print('\n' + '='*45)
print('📈 TEST RESULTS — RAISE+DiQuID Model')
print('='*45)
for name, val in zip(model.metrics_names, results):
    print(f'   {name:12s}: {val:.4f} ({val*100:.2f}%)')

# Predictions
print('\n🔍 Generating predictions...')
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = np.concatenate([y for _, y in test_ds], axis=0).astype(int).flatten()

print('\n📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Real Camera vs AI-Edited Detector', fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('✅ Confusion matrix saved to Drive.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 12: Save Final Model to Google Drive
# ─────────────────────────────────────────────────────────
import json, pickle

# Save in Keras format
model.save(MODEL_SAVE_PATH)
print(f'✅ Model saved: {MODEL_SAVE_PATH}')

# Also save as .pkl
pkl_path = MODEL_SAVE_PATH.replace('.keras', '.pkl')
try:
    with open(pkl_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'✅ Pickle saved: {pkl_path}')
except Exception as e:
    print(f'⚠️  Pickle failed (not critical): {e}')

# Save training history
history_data = {}
if history_p1: history_data['p1'] = {k: [float(v) for v in vs] for k, vs in history_p1.history.items()}
if history_p2: history_data['p2'] = {k: [float(v) for v in vs] for k, vs in history_p2.history.items()}
if history_p3: history_data['p3'] = {k: [float(v) for v in vs] for k, vs in history_p3.history.items()}

with open(HISTORY_FILE, 'w') as f:
    json.dump(history_data, f, indent=2)
print(f'✅ Training history saved: {HISTORY_FILE}')

print('\n🎉 ALL DONE!')
print(f'   Model : {MODEL_SAVE_PATH}')
print('\nNext steps:')
print('  1. Download raise_diquid_efficientnetv2b2.keras from Google Drive')
print('  2. Place it in: d:/Media Validate App/models/')
print('  3. Update backend model path to use this new model')
print('  4. Restart the backend Python API')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 14: Training History Plot
# ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

all_acc, all_val_acc   = [], []
all_loss, all_val_loss = [], []
phase_boundaries       = []

for h_obj in [history_p1, history_p2, history_p3]:
    if h_obj:
        phase_boundaries.append(len(all_acc))
        all_acc      += h_obj.history.get('accuracy', [])
        all_val_acc  += h_obj.history.get('val_accuracy', [])
        all_loss     += h_obj.history.get('loss', [])
        all_val_loss += h_obj.history.get('val_loss', [])

if not all_acc:
    print('⚠️  No history in this session (all phases resumed). Check history JSON on Drive.')
else:
    epochs = range(1, len(all_acc) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('RAISE+DiQuID Training — Real Camera vs AI-Edited', fontweight='bold')

    ax1.plot(epochs, all_acc,     'b-o', ms=3, label='Train Accuracy')
    ax1.plot(epochs, all_val_acc, 'r-o', ms=3, label='Val Accuracy')
    for b in phase_boundaries[1:]:
        ax1.axvline(x=b, color='gray', linestyle='--', alpha=0.5, label=f'Phase boundary')
    ax1.set_title('Accuracy Over Epochs')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, all_loss,     'b-o', ms=3, label='Train Loss')
    ax2.plot(epochs, all_val_loss, 'r-o', ms=3, label='Val Loss')
    for b in phase_boundaries[1:]:
        ax2.axvline(x=b, color='gray', linestyle='--', alpha=0.5)
    ax2.set_title('Loss Over Epochs')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Training plot saved: {PLOT_PATH}')